In [1]:
import os
import re
import sys
import json
import types
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

# Fix seed everywhere
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


Device : cuda
PyTorch: 2.10.0+cu128


In [12]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "split" in file.lower():
            print(os.path.join(root, file))

In [13]:
DATA_DIR = Path("/kaggle/input/datasets/varunvijay007/ildc-cleanded-2")

train_df = pd.read_csv(DATA_DIR / "train_clean_baseline1.csv")
val_df   = pd.read_csv(DATA_DIR / "val_clean_baseline1.csv")
test_df  = pd.read_csv(DATA_DIR / "test_clean_baseline1.csv")

# Drop nulls
train_df = train_df.dropna(subset=['clean_text', 'label']).reset_index(drop=True)
val_df   = val_df.dropna(subset=['clean_text', 'label']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['clean_text', 'label']).reset_index(drop=True)

print("Before stratified split:")
print(f"  Train : {len(train_df)} | "
      f"{train_df['label'].value_counts(normalize=True).to_dict()}")
print(f"  Val   : {len(val_df)}   | "
      f"{val_df['label'].value_counts(normalize=True).to_dict()}")
print(f"  Test  : {len(test_df)}  | "
      f"{test_df['label'].value_counts(normalize=True).to_dict()}")

Before stratified split:
  Train : 6111 | {0: 0.6082474226804123, 1: 0.3917525773195876}
  Val   : 1012   | {1: 0.5207509881422925, 0: 0.4792490118577075}
  Test  : 1071  | {1: 0.5518207282913166, 0: 0.4481792717086835}


# Stratified Train-Test Split

# ── Rhetorical Role Model (Pretrained Hier-BiLSTM-CRF, 84% accuracy)

In [15]:
# ══════════════════════════════════════════════════════════════════
# Pretrained rhetorical role model — Law-AI Hier-BiLSTM-CRF
# Replaces ALL keyword/rule-based role assignment (assign_role etc.)
# Files needed in /kaggle/input/rhetorical-role-model/:
#   model.tar, word2idx.json, tag2idx.json, sent2vec.bin
# ══════════════════════════════════════════════════════════════════

# ── 1. Install sent2vec (Linux / Kaggle) ──────────────────────────
import subprocess
import sys
import shutil

try:
    import sent2vec as sv
    print("sent2vec already installed")
except ImportError:

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "/kaggle/input/sent2vec-wheel/sent2vec",
            "--quiet"
        ],
        capture_output=True
    )

    if result.returncode != 0:

        # remove previous failed clone if it exists
        shutil.rmtree("/tmp/sent2vec", ignore_errors=True)

        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/epfml/sent2vec.git",
                "/tmp/sent2vec"
            ],
            check=True
        )

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "/tmp/sent2vec",
                "--quiet"
            ],
            check=True
        )

    import sent2vec as sv

print("sent2vec ready")

# ── 2. Paths ──────────────────────────────────────────────────────
ROLE_MODEL_DIR = Path("/kaggle/input/datasets/varunvijay007/rhetorical-role-models/")
ROLE_MODEL_TAR = ROLE_MODEL_DIR / "model.tar"
WORD2IDX_PATH  = ROLE_MODEL_DIR / "word2idx(2).json"
TAG2IDX_PATH   = ROLE_MODEL_DIR / "tag2idx(2).json"
SENT2VEC_PATH  = ROLE_MODEL_DIR / "sent2vec.bin"

# ── 3. Load semantic-segmentation repo model classes ─────────────
# Clone repo so we can import the model class
SEMSEG_DIR = Path("/kaggle/working/semseg")
if not SEMSEG_DIR.exists():
    subprocess.run(
        "git clone https://github.com/Law-AI/semantic-segmentation.git "
        f"{SEMSEG_DIR} -q",
        shell=True, check=True
    )
sys.path.insert(0, str(SEMSEG_DIR))

from model.Hier_BiLSTM_CRF import Hier_LSTM_CRF_Classifier

with open(WORD2IDX_PATH) as f:
    _word2idx = json.load(f)
with open(TAG2IDX_PATH) as f:
    _tag2idx = json.load(f)

_idx2tag = {v: k for k, v in _tag2idx.items()}

RR_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_ckpt = torch.load(
    str(ROLE_MODEL_TAR),
    map_location=RR_DEVICE,
    weights_only=False
)

_role_clf = Hier_LSTM_CRF_Classifier(
    n_tags       = len(_tag2idx),
    sent_emb_dim = 200,
    sos_tag_idx  = _tag2idx["<start>"],
    eos_tag_idx  = _tag2idx["<end>"],
    pad_tag_idx  = _tag2idx["<pad>"],
    vocab_size   = len(_word2idx),
    pretrained   = True,
    device       = RR_DEVICE
)

_role_clf.load_state_dict(_ckpt["state_dict"])
_role_clf = _role_clf.to(RR_DEVICE)
_role_clf.eval()

print("RR device:", RR_DEVICE)
print("Model device:", next(_role_clf.parameters()).device)
print("Hier-BiLSTM-CRF role model loaded")

# ── 4. Load sent2vec for sentence embedding ───────────────────────
print("Loading sent2vec model (~2 GB, takes ~60s)...")
_s2v = sv.Sent2vecModel()
_s2v.load_model(str(SENT2VEC_PATH))
print("sent2vec loaded")

# ── 5. Sentence splitter (spacy — keep same interface) ───────────
import spacy
nlp = spacy.load("en_core_web_sm")

def segment_sentences(text: str, max_sents: int = 64):
    doc = nlp(text[:50000])
    return [
        s.text.strip() for s in doc.sents
        if len(s.text.strip()) > 20
    ][:max_sents]

# ── 6. Role → weight mapping (7 labels from the trained model) ───
# Labels: Facts, Argument, Ratio of the decision,
#         Statute, Precedent, Ruling by Present Court,
#         Ruling by Lower Court
ROLE_WEIGHTS = {
    "Facts"                    : 0.6,
    "Argument"                 : 0.7,
    "Ratio of the decision"    : 1.0,   # highest — key reasoning
    "Statute"                  : 0.8,
    "Precedent"                : 0.8,
    "Ruling by Present Court"  : 1.0,   # highest — final ruling
    "Ruling by Lower Court"    : 0.5,
    # fallback
    "<pad>"                    : 0.1,
    "<start>"                  : 0.1,
    "<end>"                    : 0.1,
}

# ── 7. Core function: predict roles for a list of sentences ───────
@torch.no_grad()
def predict_roles(sentences: list) -> list:
    """
    Run the pretrained Hier-BiLSTM-CRF model on a list of sentences.
    Returns a list of string role labels, one per sentence.
    """
    if not sentences:
        return []
    # Embed each sentence with sent2vec → [n_sents, 200]
    doc_x = [
        _s2v.embed_sentence(s).flatten().tolist()[:200]
        for s in sentences
    ]
    # Model expects a list-of-docs; pass single doc wrapped in list
    pred_indices = _role_clf([doc_x])[0]   # list of int indices
    return [_idx2tag.get(idx, "Facts") for idx in pred_indices]

# ── 8. Quick verification ─────────────────────────────────────────
_test_sents = [
    "The appellant filed the petition in the Supreme Court in 1998.",
    "Section 302 IPC prescribes punishment for murder.",
    "In State v. Mohan this Court held that intent must be proven.",
    "The appeal is therefore allowed and the conviction set aside.",
]
_roles = predict_roles(_test_sents)
print("\nRole prediction verification:")
for s, r in zip(_test_sents, _roles):
    print(f"  [{r:<26}]  {s[:70]}")
print("\n✓ Pretrained rhetorical role model active — rule-based system REMOVED")


Cloning into '/tmp/sent2vec'...


sent2vec ready
RR device: cuda
Model device: cuda:0
Hier-BiLSTM-CRF role model loaded
Loading sent2vec model (~2 GB, takes ~60s)...
sent2vec loaded

Role prediction verification:
  [Facts                     ]  The appellant filed the petition in the Supreme Court in 1998.
  [Ruling by Lower Court     ]  Section 302 IPC prescribes punishment for murder.
  [Ruling by Lower Court     ]  In State v. Mohan this Court held that intent must be proven.
  [Facts                     ]  The appeal is therefore allowed and the conviction set aside.

✓ Pretrained rhetorical role model active — rule-based system REMOVED


# ── CELL 5: Build role weight tensor per document ─────────────
# This is the ACTIVE novelty — role weights passed to model.
# For each tokenized document, we build a [seq_len] weight tensor
# where each token position gets the weight of its sentence's role.

In [16]:
LEGALBERT_MODEL = 'law-ai/InLegalBERT'

tokenizer = AutoTokenizer.from_pretrained(LEGALBERT_MODEL)

print("Tokenizer loaded")

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded


In [19]:
sample_text = train_df['clean_text'].iloc[0]

sample_weights = build_token_role_weights(
    sample_text,
    tokenizer
)

print("Role weights shape:", sample_weights.shape)
print("First 20 weights:", sample_weights[:20])
print("Mean weight:", sample_weights.mean().item())

Role weights shape: torch.Size([512])
First 20 weights: tensor([0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000,
        0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000,
        0.6000, 0.6000])
Mean weight: 0.717578113079071


In [18]:
def build_token_role_weights(text: str, tokenizer,
                              max_len: int = 512) -> torch.Tensor:
    """
    Build a [max_len] weight tensor where each token position gets
    the weight of its sentence role — predicted by the pretrained
    Hier-BiLSTM-CRF model (NOT rule-based).
    """
    weights   = torch.ones(max_len, dtype=torch.float32)
    sentences = segment_sentences(text)
    if not sentences:
        return weights

    # Predict roles with pretrained model
    roles = predict_roles(sentences)   # list[str], one per sentence

    enc = tokenizer(
        text[:3000],
        truncation             = True,
        max_length             = max_len,
        return_offsets_mapping = True,
        padding                = 'max_length',
        return_tensors         = 'pt'
    )
    offsets = enc['offset_mapping'][0].tolist()   # [(start, end), ...]

    # Build character spans for each sentence
    char_pos    = 0
    sent_spans  = []
    for sent, role in zip(sentences, roles):
        start  = text.find(sent, char_pos)
        if start == -1:
            start = char_pos
        end    = start + len(sent)
        weight = ROLE_WEIGHTS.get(role, 0.5)
        sent_spans.append((start, end, weight))
        char_pos = end

    # Map token → sentence weight
    for tok_idx, (tok_start, tok_end) in enumerate(offsets):
        if tok_idx >= max_len:
            break
        if tok_start == tok_end == 0 and tok_idx > 0:
            continue   # padding token
        for (s_start, s_end, weight) in sent_spans:
            if tok_start >= s_start and tok_end <= s_end:
                weights[tok_idx] = weight
                break

    return weights   # [max_len]


print("build_token_role_weights defined — using pretrained model roles")


build_token_role_weights defined — using pretrained model roles


# ── CELL 7: Dataset class

In [20]:
MAX_LEN   = 512
MAX_CHARS = 3000

class LegalBERTDataset(Dataset):
    """
    Dataset that builds role-weight tensors per document using
    the pretrained rhetorical role model.
    """
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

        texts = df['clean_text'].str[:MAX_CHARS].tolist()
        self.encodings = tokenizer(
            texts,
            truncation     = True,
            padding        = 'max_length',
            max_length     = max_len,
            return_tensors = 'pt'
        )
        self.labels = torch.tensor(df['label'].tolist(), dtype=torch.long)

        print(f"Precomputing role weights for {len(df)} documents...")
        self.role_weights = torch.stack([
            build_token_role_weights(text, tokenizer, max_len)
            for text in tqdm(texts)
        ])   # [N, max_len]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            {k: v[idx] for k, v in self.encodings.items()},
            self.role_weights[idx],
            self.labels[idx]
        )

print("Building train dataset...")
train_bert_ds = LegalBERTDataset(train_df, tokenizer)
print("Building val dataset...")
val_bert_ds   = LegalBERTDataset(val_df,   tokenizer)
print("Building test dataset...")
test_bert_ds  = LegalBERTDataset(test_df,  tokenizer)

# IMPROVEMENT: larger batch for val/test (no gradient, fits in GPU)
train_bert_loader = DataLoader(
    train_bert_ds, batch_size=8, shuffle=True,
    worker_init_fn=lambda _: np.random.seed(SEED)
)
val_bert_loader = DataLoader(val_bert_ds,   batch_size=16, shuffle=False)
test_bert_loader = DataLoader(test_bert_ds, batch_size=16, shuffle=False)

print(f"\nTrain batches: {len(train_bert_loader)}")
print(f"Val   batches: {len(val_bert_loader)}")
print(f"Test  batches: {len(test_bert_loader)}")

sample_enc, sample_rw, sample_label = train_bert_ds[0]
print(f"\ninput_ids shape    : {sample_enc['input_ids'].shape}")
print(f"role_weights shape : {sample_rw.shape}")
unique_w = sample_rw.unique().tolist()
print(f"Unique role weights: {unique_w}")
print("Check: weights should include values like 0.6, 0.7, 0.8, 1.0")
print("Check: NOT all 1.0 — if all 1.0 the pretrained model is not active")


Building train dataset...
Precomputing role weights for 6111 documents...


100%|██████████| 6111/6111 [08:02<00:00, 12.66it/s]


Building val dataset...
Precomputing role weights for 1012 documents...


100%|██████████| 1012/1012 [01:15<00:00, 13.33it/s]


Building test dataset...
Precomputing role weights for 1071 documents...


100%|██████████| 1071/1071 [01:21<00:00, 13.15it/s]


Train batches: 764
Val   batches: 64
Test  batches: 67

input_ids shape    : torch.Size([512])
role_weights shape : torch.Size([512])
Unique role weights: [0.6000000238418579, 0.699999988079071, 0.800000011920929, 1.0]
Check: weights should include values like 0.6, 0.7, 0.8, 1.0
Check: NOT all 1.0 — if all 1.0 the pretrained model is not active


save the weights

In [21]:
torch.save(
    train_bert_ds.role_weights,
    "/kaggle/working/train_role_weights.pt"
)

torch.save(
    val_bert_ds.role_weights,
    "/kaggle/working/val_role_weights.pt"
)

torch.save(
    test_bert_ds.role_weights,
    "/kaggle/working/test_role_weights.pt"
)

print("Role weights saved")

Role weights saved


# Model Definition
"""
    InLegalBERT with rhetorical role-weighted pooling.
    NOVELTY: instead of CLS token, uses weighted mean pooling
    where each token's contribution is scaled by its sentence's
    rhetorical role importance (Ruling=1.0, Preamble=0.2 etc.)
    """

"""
        role_weights: [batch, seq_len] — weight per token position
                      based on rhetorical role of its sentence.
        Uses role-weighted mean pooling instead of CLS token.
        """

In [22]:
class RoleWeightedLegalBERT(nn.Module):
    """
    InLegalBERT with rhetorical role-weighted pooling.
    IMPROVEMENT: added LayerNorm on pooled output + wider classifier
    for better representation capacity.
    """
    def __init__(self, model_name, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        hidden_size     = self.bert.config.hidden_size   # 768

        # IMPROVEMENT: LayerNorm stabilises role-weighted pooled vector
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout    = nn.Dropout(dropout)

        # IMPROVEMENT: 2-layer classifier instead of 1 linear layer
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, role_weights):
        outputs = self.bert(
            input_ids      = input_ids,
            attention_mask = attention_mask
        )
        hidden = outputs.last_hidden_state   # [batch, seq_len, 768]

        # Role-weighted mean pooling
        mask = attention_mask.unsqueeze(-1).float()    # [batch, seq_len, 1]
        rw   = role_weights.unsqueeze(-1)              # [batch, seq_len, 1]

        weighted_sum = (hidden * rw * mask).sum(dim=1)          # [batch, 768]
        weight_total = (rw * mask).sum(dim=1).clamp(min=1e-9)   # [batch, 1]
        pooled       = weighted_sum / weight_total               # [batch, 768]

        # IMPROVEMENT: normalise before classification
        pooled = self.layer_norm(pooled)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

transformer_model = RoleWeightedLegalBERT(LEGALBERT_MODEL).to(DEVICE)
print("RoleWeightedLegalBERT built (with LayerNorm + 2-layer classifier)")
total = sum(p.numel() for p in transformer_model.parameters())
print(f"Total parameters: {total:,}")

test_ids  = torch.randint(0, 1000, (2, 512)).to(DEVICE)
test_mask = torch.ones(2, 512, dtype=torch.long).to(DEVICE)
test_rw   = torch.rand(2, 512).to(DEVICE)
test_out  = transformer_model(test_ids, test_mask, test_rw)
print(f"Test output shape: {test_out.shape}")   # [2, 2]


pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/534M [00:00<?, ?B/s]

RoleWeightedLegalBERT built (with LayerNorm + 2-layer classifier)
Total parameters: 109,681,154
Test output shape: torch.Size([2, 2])


In [24]:
label_counts  = train_df['label'].value_counts().sort_index()
total_samples = len(train_df)
class_weights = torch.tensor(
    [total_samples / (2 * label_counts[i]) for i in range(2)],
    dtype=torch.float32
).to(DEVICE)
print(f"Class weights: {class_weights.tolist()}")
print("Check: both weights should be close to 1.0 after stratification")

criterion_bert = nn.CrossEntropyLoss(weight=class_weights)

Class weights: [0.8220338821411133, 1.2763158082962036]
Check: both weights should be close to 1.0 after stratification


# ── CELL 10: Training loop — novelty ACTIVE

In [25]:
from torch.optim import AdamW

# IMPROVEMENT: more epochs (7 vs 5) — with 80% train data
# IMPROVEMENT: label_smoothing reduces overconfidence
EPOCHS_BERT  = 7
LR_BERT      = 2e-5
LABEL_SMOOTH = 0.1

optimizer_bert = AdamW(
    [
        # Lower LR for BERT encoder layers — prevents catastrophic forgetting
        {"params": transformer_model.bert.parameters(),        "lr": LR_BERT},
        # Higher LR for classifier head — trains faster from scratch
        {"params": transformer_model.classifier.parameters(),  "lr": LR_BERT * 5},
        {"params": transformer_model.layer_norm.parameters(),  "lr": LR_BERT * 5},
    ],
    weight_decay = 0.01
)

total_steps  = len(train_bert_loader) * EPOCHS_BERT
warmup_steps = total_steps // 10

scheduler_bert = get_linear_schedule_with_warmup(
    optimizer_bert,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

# IMPROVEMENT: label smoothing + class weights combined
criterion_bert = nn.CrossEntropyLoss(
    weight        = class_weights,
    label_smoothing = LABEL_SMOOTH
)

best_val_f1    = 0.0
best_bert_path = OUTPUT_DIR / "best_inlegalbert.pt"
bert_train_losses = []
bert_val_f1s      = []

# IMPROVEMENT: early stopping patience
PATIENCE      = 3
patience_ctr  = 0

print("Training RoleWeightedLegalBERT...")
print(f"  Epochs        : {EPOCHS_BERT}")
print(f"  LR (BERT)     : {LR_BERT}")
print(f"  LR (head)     : {LR_BERT * 5}")
print(f"  Label smooth  : {LABEL_SMOOTH}")
print(f"  Total steps   : {total_steps}")
print(f"  Warmup steps  : {warmup_steps}")
print(f"  Early stop    : patience={PATIENCE}\n")

for epoch in range(EPOCHS_BERT):
    transformer_model.train()
    total_loss = 0

    for batch_enc, batch_rw, batch_labels in tqdm(
        train_bert_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS_BERT} [train]"
    ):
        batch_enc    = {k: v.to(DEVICE) for k, v in batch_enc.items()}
        batch_rw     = batch_rw.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)

        logits = transformer_model(
            input_ids      = batch_enc['input_ids'],
            attention_mask = batch_enc['attention_mask'],
            role_weights   = batch_rw
        )

        loss = criterion_bert(logits, batch_labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            transformer_model.parameters(), max_norm=1.0
        )
        optimizer_bert.step()
        scheduler_bert.step()
        optimizer_bert.zero_grad()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_bert_loader)
    bert_train_losses.append(avg_loss)

    transformer_model.eval()
    val_preds, val_true = [], []

    with torch.no_grad():
        for batch_enc, batch_rw, batch_labels in tqdm(
            val_bert_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS_BERT} [val]  "
        ):
            batch_enc = {k: v.to(DEVICE) for k, v in batch_enc.items()}
            batch_rw  = batch_rw.to(DEVICE)

            logits = transformer_model(
                input_ids      = batch_enc['input_ids'],
                attention_mask = batch_enc['attention_mask'],
                role_weights   = batch_rw
            )
            preds = logits.argmax(dim=-1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(batch_labels.numpy())

    val_f1 = f1_score(val_true, val_preds, average='macro')
    bert_val_f1s.append(val_f1)

    print(f"\nEpoch {epoch+1} — Loss: {avg_loss:.4f}  Val Macro F1: {val_f1:.4f}")
    print(classification_report(val_true, val_preds,
                                  target_names=['Rejected', 'Accepted']))

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        patience_ctr = 0
        torch.save(transformer_model.state_dict(), best_bert_path)
        print(f"  → Best model saved (Val F1: {val_f1:.4f})")
    else:
        patience_ctr += 1
        print(f"  No improvement. Patience: {patience_ctr}/{PATIENCE}")
        if patience_ctr >= PATIENCE:
            print("  Early stopping triggered.")
            break

print(f"\nBest validation Macro F1: {best_val_f1:.4f}")


Training RoleWeightedLegalBERT...
  Epochs        : 7
  LR (BERT)     : 2e-05
  LR (head)     : 0.0001
  Label smooth  : 0.1
  Total steps   : 5348
  Warmup steps  : 534
  Early stop    : patience=3



Epoch 1/7 [val]  : 100%|██████████| 64/64 [00:39<00:00,  1.63it/s]



Epoch 1 — Loss: 0.7081  Val Macro F1: 0.4913
              precision    recall  f1-score   support

    Rejected       0.51      0.90      0.65       485
    Accepted       0.70      0.22      0.33       527

    accuracy                           0.54      1012
   macro avg       0.60      0.56      0.49      1012
weighted avg       0.61      0.54      0.48      1012

  → Best model saved (Val F1: 0.4913)


Epoch 2/7 [val]  : 100%|██████████| 64/64 [00:39<00:00,  1.63it/s]



Epoch 2 — Loss: 0.6982  Val Macro F1: 0.5637
              precision    recall  f1-score   support

    Rejected       0.66      0.33      0.44       485
    Accepted       0.58      0.84      0.69       527

    accuracy                           0.60      1012
   macro avg       0.62      0.59      0.56      1012
weighted avg       0.62      0.60      0.57      1012

  → Best model saved (Val F1: 0.5637)


...


Epoch 4 — Loss: 0.5118  Val Macro F1: 0.5742
              precision    recall  f1-score   support

    Rejected       0.55      0.70      0.61       485
    Accepted       0.63      0.46      0.53       527

    accuracy                           0.58      1012
   macro avg       0.59      0.58      0.57      1012
weighted avg       0.59      0.58      0.57      1012

  No improvement. Patience: 1/3


Epoch 5/7 [val]  : 100%|██████████| 64/64 [00:39<00:00,  1.63it/s]



Epoch 5 — Loss: 0.3862  Val Macro F1: 0.5899
              precision    recall  f1-score   support

    Rejected       0.57      0.60      0.59       485
    Accepted       0.61      0.58      0.59       527

    accuracy                           0.59      1012
   macro avg       0.59      0.59      0.59      1012
weighted avg       0.59      0.59      0.59      1012

  No improvement. Patience: 2/3


Epoch 6/7 [val]  : 100%|██████████| 64/64 [00:39<00:00,  1.63it/s]


Epoch 6 — Loss: 0.3043  Val Macro F1: 0.5909
              precision    recall  f1-score   support

    Rejected       0.57      0.62      0.59       485
    Accepted       0.62      0.56      0.59       527

    accuracy                           0.59      1012
   macro avg       0.59      0.59      0.59      1012
weighted avg       0.59      0.59      0.59      1012

  No improvement. Patience: 3/3
  Early stopping triggered.

Best validation Macro F1: 0.6394


# ── CELL 8: MC Dropout — InLegalBERT

In [26]:
def enable_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

def mc_dropout_predict_bert(model, data_loader,
                              n_passes=20, device=DEVICE):
    """
    MC Dropout inference — runs n_passes forward passes
    with dropout active. Returns mean prediction,
    confidence, and uncertainty per sample.
    """
    model.load_state_dict(torch.load(best_bert_path, weights_only=False))
    model.eval()
    enable_dropout(model)   # keep dropout ON

    all_mean_probs = []
    all_std_probs  = []
    all_labels     = []

    with torch.no_grad():
        for batch_enc, batch_rw, batch_labels in tqdm(
            data_loader, desc="MC Dropout — InLegalBERT"
        ):
            batch_enc = {k: v.to(device) for k, v in batch_enc.items()}
            batch_rw  = batch_rw.to(device)
            pass_probs = []

            for _ in range(n_passes):
                logits = model(
                    input_ids      = batch_enc['input_ids'],
                    attention_mask = batch_enc['attention_mask'],
                    role_weights   = batch_rw   # role weights in MC too
                )
                probs = torch.softmax(logits, dim=-1)
                pass_probs.append(probs.cpu())

            pass_probs = torch.stack(pass_probs)  # [n_passes, batch, 2]
            mean_prob  = pass_probs.mean(dim=0)   # [batch, 2]
            std_prob   = pass_probs.std(dim=0)    # [batch, 2]

            all_mean_probs.append(mean_prob)
            all_std_probs.append(std_prob)
            all_labels.extend(batch_labels.numpy())

    mean_probs  = torch.cat(all_mean_probs, dim=0)
    std_probs   = torch.cat(all_std_probs,  dim=0)
    predictions = mean_probs.argmax(dim=-1).numpy()
    confidence  = mean_probs.max(dim=-1).values.numpy()
    uncertainty = std_probs.max(dim=-1).values.numpy()

    return predictions, confidence, uncertainty, np.array(all_labels)

print("Running MC Dropout on test set...")
bert_preds, bert_conf, bert_unc, bert_true = mc_dropout_predict_bert(
    transformer_model, test_bert_loader, n_passes=20
)

print("\n" + "="*60)
print("InLegalBERT (Role-Weighted) — Test Results")
print("="*60)
print(classification_report(
    bert_true, bert_preds,
    target_names=['Rejected', 'Accepted']
))
print(f"Accuracy       : {accuracy_score(bert_true, bert_preds):.4f}")
print(f"Macro F1       : {f1_score(bert_true, bert_preds, average='macro'):.4f}")
print(f"Avg confidence : {bert_conf.mean():.4f}")
print(f"Avg uncertainty: {bert_unc.mean():.4f}")

# Verification
print("\nMC Dropout verification:")
print(f"  High confidence (>0.85) : "
      f"{(bert_conf > 0.85).sum()} / {len(bert_conf)}")
print(f"  High uncertainty (>0.15): "
      f"{(bert_unc > 0.15).sum()} / {len(bert_unc)}")
print("  Check: uncertainty should vary — not all same value")
print("  Check: high uncertainty cases = model is unsure = harder cases")


Running MC Dropout on test set...


MC Dropout — InLegalBERT: 100%|██████████| 67/67 [14:03<00:00, 12.58s/it]


InLegalBERT (Role-Weighted) — Test Results
              precision    recall  f1-score   support

    Rejected       0.53      0.41      0.46       480
    Accepted       0.59      0.70      0.64       591

    accuracy                           0.57      1071
   macro avg       0.56      0.56      0.55      1071
weighted avg       0.56      0.57      0.56      1071

Accuracy       : 0.5714
Macro F1       : 0.5525
Avg confidence : 0.6413
Avg uncertainty: 0.0454

MC Dropout verification:
  High confidence (>0.85) : 33 / 1071
  High uncertainty (>0.15): 0 / 1071
  Check: uncertainty should vary — not all same value
  Check: high uncertainty cases = model is unsure = harder cases


In [34]:
bert_results_df = pd.DataFrame({
    'true_label' : bert_true,
    'prediction' : bert_preds,
    'confidence' : bert_conf,
    'uncertainty': bert_unc,
    'model'      : 'InLegalBERT_RoleWeighted'
})
bert_results_df.to_csv(
    OUTPUT_DIR / "bert_test_results.csv", index=False
)
print("\nBERT results saved.")
print("Continue with BiLSTM cells from original Notebook 2 (Cell 9 onwards)")



BERT results saved.
Continue with BiLSTM cells from original Notebook 2 (Cell 9 onwards)


BRANCH 2: BiLSTM + ATTENTION with NER-type embeddings
# Input : clean text + NER entity type embeddings
# Novelty: entity-aware input augmentation (NOT masking)

In [35]:
from transformers import pipeline
print("Loading NER model...")
ner_pipe = pipeline(
    'ner',
    model                = 'dslim/bert-base-NER',
    aggregation_strategy = 'simple',
    device               = 0 if torch.cuda.is_available() else -1
)
print("NER model loaded")

Loading NER model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER model loaded


# Map NER entity group labels → integer IDs

In [36]:
ENTITY_TYPE_MAP = {
    'O'    : 0,   # no entity
    'PER'  : 1,   # person
    'ORG'  : 2,   # organisation / court
    'LOC'  : 3,   # location
    'MISC' : 4,   # miscellaneous (statutes, case citations, etc.)
}
NUM_ENTITY_TYPES = len(ENTITY_TYPE_MAP) 

"""
    For each token position in the tokenized text,
    assign an entity type ID using NER span offsets.

    Args:
        text     : raw sentence string
        tokenizer: InLegalBERT tokenizer (for offset mapping)
        max_len  : must match the max_len used when encoding
                   this sentence — always pass 128 explicitly

    Returns:
        entity_ids: LongTensor [max_len]
    """

In [42]:
def get_token_entity_types(text: str, tokenizer,
                            max_len: int = 128) -> torch.Tensor:
    
    entity_ids = torch.zeros(max_len, dtype=torch.long)

    try:
        # NER runs on the same text slice used for tokenization
        entities = ner_pipe(text[:512])

        # Tokenize with offset mapping so we can align spans → tokens
        enc = tokenizer(
            text[:512],
            truncation            = True,
            max_length            = max_len,
            return_offsets_mapping= True,
            padding               = 'max_length'
        )
        offsets = enc['offset_mapping']   # list of (start, end) per token

        for ent in entities:
            ent_start = ent['start']
            ent_end   = ent['end']
            ent_type  = ENTITY_TYPE_MAP.get(ent['entity_group'], 0)

            for tok_idx, (tok_start, tok_end) in enumerate(offsets):
                # Skip special tokens (offset = (0,0) or None)
                if tok_start is None or tok_end is None:
                    continue
                if tok_start == 0 and tok_end == 0:
                    continue
                # Token is fully inside the entity character span
                if tok_start >= ent_start and tok_end <= ent_end:
                    entity_ids[tok_idx] = ent_type

    except Exception:
        pass   # return all-zeros if NER fails on this sentence

    return entity_ids


# ── CELL 10: Build sentence-level embeddings for BiLSTM

# Each document → sequence of sentence embeddings
# Each sentence embedding = InLegalBERT CLS [768] + entity signal [16]
#                         = [784] total

"""
    Encode one sentence with InLegalBERT and inject entity-type signal.

    Pipeline:
      1. Tokenise sentence → [max_len] tokens
      2. InLegalBERT CLS token → [768]
      3. NER span → per-token entity type IDs → embedding table → [max_len, 16]
      4. Mean-pool entity embeddings over entity tokens → [16]
      5. Concatenate: [768] + [16] = [784]

    Returns: FloatTensor [784]
    """

"""
    Build a padded sequence of sentence embeddings for one document.

    Returns:
        seq_tensor  : FloatTensor [max_sents, 784]   (padded with zeros)
        role_w_tensor: FloatTensor [max_sents]         (role weight per sentence)
        valid_count : int — number of real (non-fragment) sentences encoded
    """

In [43]:
sentence_encoder = AutoModel.from_pretrained(LEGALBERT_MODEL).to(DEVICE)
sentence_encoder.eval()

entity_embedding_layer = nn.Embedding(
    NUM_ENTITY_TYPES, 16, padding_idx=0
).to(DEVICE)

SENT_MAX_LEN = 128

def get_sentence_embedding_with_entities(
    sentence, tokenizer, encoder, entity_emb_layer, device,
    max_len=SENT_MAX_LEN
) -> torch.Tensor:
    enc = tokenizer(
        sentence,
        truncation    = True,
        max_length    = max_len,
        padding       = 'max_length',
        return_tensors= 'pt'
    ).to(device)

    with torch.no_grad():
        out = encoder(**enc)
        cls = out.last_hidden_state[:, 0, :]   # [1, 768]

    ent_ids  = get_token_entity_types(sentence, tokenizer, max_len=max_len).to(device)
    ent_embs = entity_emb_layer(ent_ids)

    non_zero = (ent_ids > 0).float().unsqueeze(-1)
    if non_zero.sum() > 0:
        ent_signal = (ent_embs * non_zero).sum(dim=0) / non_zero.sum()
    else:
        ent_signal = torch.zeros(16, device=device)

    return torch.cat([cls.squeeze(0), ent_signal], dim=-1)   # [784]


def build_document_sequence(
    text, tokenizer, encoder, entity_emb_layer, device, max_sents=64
) -> tuple:
    """
    Build sentence-level sequence using PRETRAINED role model for weights.
    Replaces the old assign_role() / ROLE_WEIGHTS rule-based approach.
    """
    sentences = segment_sentences(text, max_sents)
    EMBED_DIM = 768 + 16   # 784

    seq_tensor    = torch.zeros(max_sents, EMBED_DIM)
    role_w_tensor = torch.zeros(max_sents)
    valid_count   = 0

    if not sentences:
        return seq_tensor, role_w_tensor, valid_count

    # ── Pretrained model predicts roles for all sentences at once ──
    roles = predict_roles(sentences)   # list[str]

    for i, (sent, role) in enumerate(zip(sentences, roles)):
        weight = ROLE_WEIGHTS.get(role, 0.5)
        emb    = get_sentence_embedding_with_entities(
            sent, tokenizer, encoder, entity_emb_layer, device
        )
        if valid_count < max_sents:
            seq_tensor[valid_count]    = emb.detach().cpu()
            role_w_tensor[valid_count] = weight
            valid_count += 1

    return seq_tensor, role_w_tensor, valid_count


# Verification
print("Testing build_document_sequence with pretrained role model...")
sample_text = train_df['clean_text'].iloc[0]
seq, role_w, n_valid = build_document_sequence(
    sample_text, tokenizer, sentence_encoder, entity_embedding_layer, DEVICE
)
print(f"Sequence shape       : {seq.shape}")
print(f"Role weights shape   : {role_w.shape}")
print(f"Valid sentences      : {n_valid}")
print(f"Unique role weights  : {role_w[:n_valid].unique().tolist()}")
print(f"Any NaN in sequence  : {torch.isnan(seq).any().item()}")
print("Check: unique weights should reflect model predictions (NOT all same value)")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Testing build_document_sequence with pretrained role model...
Sequence shape       : torch.Size([64, 784])
Role weights shape   : torch.Size([64])
Valid sentences      : 59
Unique role weights  : [0.6000000238418579, 0.699999988079071, 0.800000011920929, 1.0]
Any NaN in sequence  : False
Check: unique weights should reflect model predictions (NOT all same value)


# CELL 11: Precompute and save all sequence embeddings

"""
    Precompute sentence-sequence embeddings for every document
    in df and save to disk as a .pt file.

    If the file already exists (e.g. second run), load and return it.
    """

In [45]:
def precompute_sequences(df, split_name, tokenizer,
                          encoder, entity_emb_layer,
                          device, max_sents=64):
    
    save_path = OUTPUT_DIR / f"{split_name}_sequences.pt"

    if save_path.exists():
        print(f"{split_name} sequences already cached — loading...")
        data = torch.load(save_path)
        return data['sequences'], data['role_weights'], data['labels']

    sequences    = []
    role_weights = []
    labels       = []

    print(f"Precomputing {split_name} sequences ({len(df)} docs)...")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        seq, role_w, _ = build_document_sequence(
            row['clean_text'], tokenizer, encoder,
            entity_emb_layer, device, max_sents
        )
        sequences.append(seq)
        role_weights.append(role_w)
        labels.append(int(row['label']))

    sequences    = torch.stack(sequences)      # [N, max_sents, 784]
    role_weights = torch.stack(role_weights)   # [N, max_sents]
    labels       = torch.tensor(labels, dtype=torch.long)

    torch.save({
        'sequences'   : sequences,
        'role_weights': role_weights,
        'labels'      : labels
    }, save_path)
    print(f"Saved {split_name} sequences → {save_path}")
    return sequences, role_weights, labels


train_seqs, train_rw, train_seq_labels = precompute_sequences(
    train_df, 'train', tokenizer,
    sentence_encoder, entity_embedding_layer, DEVICE
)
val_seqs, val_rw, val_seq_labels = precompute_sequences(
    val_df, 'val', tokenizer,
    sentence_encoder, entity_embedding_layer, DEVICE
)
test_seqs, test_rw, test_seq_labels = precompute_sequences(
    test_df, 'test', tokenizer,
    sentence_encoder, entity_embedding_layer, DEVICE
)

print(f"\nTrain sequences shape: {train_seqs.shape}")   # [N_train, 64, 784]
print(f"Val   sequences shape: {val_seqs.shape}")
print(f"Test  sequences shape: {test_seqs.shape}")

Precomputing train sequences (6111 docs)...


100%|██████████| 6111/6111 [2:53:39<00:00,  1.71s/it]  


Saved train sequences → /kaggle/working/train_sequences.pt
Precomputing val sequences (1012 docs)...


100%|██████████| 1012/1012 [27:23<00:00,  1.62s/it]


Saved val sequences → /kaggle/working/val_sequences.pt
Precomputing test sequences (1071 docs)...


100%|██████████| 1071/1071 [27:36<00:00,  1.55s/it]


Saved test sequences → /kaggle/working/test_sequences.pt

Train sequences shape: torch.Size([6111, 64, 784])
Val   sequences shape: torch.Size([1012, 64, 784])
Test  sequences shape: torch.Size([1071, 64, 784])


In [46]:
entity_emb_save_path = OUTPUT_DIR / "entity_emb_layer.pt"
torch.save(entity_embedding_layer.state_dict(), entity_emb_save_path)
print(f"Entity embedding layer saved → {entity_emb_save_path}")
print("If you reload cached sequences later, also load this file:")
print("  entity_embedding_layer.load_state_dict("
      "torch.load('entity_emb_layer.pt'))")

Entity embedding layer saved → /kaggle/working/entity_emb_layer.pt
If you reload cached sequences later, also load this file:
  entity_embedding_layer.load_state_dict(torch.load('entity_emb_layer.pt'))


In [47]:
# =========================
# CELL 12 — LOAD SEQUENCES
# =========================

import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path



# -------------------------------------------------
# OPTION 2:
# If files are inside datasets folder
# -------------------------------------------------

DATASET_DIR = Path("/kaggle/working")

TRAIN_PATH = DATASET_DIR / "train_sequences.pt"
VAL_PATH   = DATASET_DIR / "val_sequences.pt"
TEST_PATH  = DATASET_DIR / "test_sequences.pt"

# -------------------------------------------------
# LOAD FILES
# -------------------------------------------------

print("Loading precomputed sequence files...")

train_data = torch.load(TRAIN_PATH, map_location="cpu")
val_data   = torch.load(VAL_PATH,   map_location="cpu")
test_data  = torch.load(TEST_PATH,  map_location="cpu")

print("Files loaded successfully.")

# -------------------------------------------------
# EXTRACT COMPONENTS
# -------------------------------------------------

train_seqs       = train_data['sequences']
train_rw         = train_data['role_weights']
train_seq_labels = train_data['labels']

val_seqs         = val_data['sequences']
val_rw           = val_data['role_weights']
val_seq_labels   = val_data['labels']

test_seqs        = test_data['sequences']
test_rw          = test_data['role_weights']
test_seq_labels  = test_data['labels']

# -------------------------------------------------
# VERIFY SHAPES
# -------------------------------------------------

print("\nTrain Shapes:")
print("Sequences    :", train_seqs.shape)
print("Role Weights :", train_rw.shape)
print("Labels       :", train_seq_labels.shape)

print("\nValidation Shapes:")
print("Sequences    :", val_seqs.shape)
print("Role Weights :", val_rw.shape)
print("Labels       :", val_seq_labels.shape)

print("\nTest Shapes:")
print("Sequences    :", test_seqs.shape)
print("Role Weights :", test_rw.shape)
print("Labels       :", test_seq_labels.shape)

# -------------------------------------------------
# DATASET CLASS
# -------------------------------------------------

class BiLSTMDataset(Dataset):
    def __init__(self, sequences, role_weights, labels):
        self.sequences    = sequences
        self.role_weights = role_weights
        self.labels       = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.sequences[idx],
            self.role_weights[idx],
            self.labels[idx]
        )

# -------------------------------------------------
# CREATE DATASETS
# -------------------------------------------------

train_bilstm_ds = BiLSTMDataset(
    train_seqs,
    train_rw,
    train_seq_labels
)

val_bilstm_ds = BiLSTMDataset(
    val_seqs,
    val_rw,
    val_seq_labels
)

test_bilstm_ds = BiLSTMDataset(
    test_seqs,
    test_rw,
    test_seq_labels
)

# -------------------------------------------------
# CREATE DATALOADERS
# -------------------------------------------------

train_bilstm_loader = DataLoader(
    train_bilstm_ds,
    batch_size=16,
    shuffle=True
)

val_bilstm_loader = DataLoader(
    val_bilstm_ds,
    batch_size=32,
    shuffle=False
)

test_bilstm_loader = DataLoader(
    test_bilstm_ds,
    batch_size=32,
    shuffle=False
)

# -------------------------------------------------
# VERIFY BATCH COUNTS
# -------------------------------------------------

print("\nDataLoaders Created Successfully.")

print(f"BiLSTM train batches: {len(train_bilstm_loader)}")
print(f"BiLSTM val   batches: {len(val_bilstm_loader)}")
print(f"BiLSTM test  batches: {len(test_bilstm_loader)}")

Loading precomputed sequence files...
Files loaded successfully.

Train Shapes:
Sequences    : torch.Size([6111, 64, 784])
Role Weights : torch.Size([6111, 64])
Labels       : torch.Size([6111])

Validation Shapes:
Sequences    : torch.Size([1012, 64, 784])
Role Weights : torch.Size([1012, 64])
Labels       : torch.Size([1012])

Test Shapes:
Sequences    : torch.Size([1071, 64, 784])
Role Weights : torch.Size([1071, 64])
Labels       : torch.Size([1071])

DataLoaders Created Successfully.
BiLSTM train batches: 382
BiLSTM val   batches: 32
BiLSTM test  batches: 34


In [49]:
class BiLSTMAttentionClassifier(nn.Module):
    """
    IMPROVEMENT: hidden_dim 256→384, added highway connection on context,
    role-weight bias on attention is preserved.
    """
    def __init__(self, input_dim=784, hidden_dim=384,
                 num_classes=2, num_layers=2, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if num_layers > 1 else 0.0
        )
        # IMPROVEMENT: attention over concatenated fwd+bwd = hidden_dim*2
        self.attention  = nn.Linear(hidden_dim * 2, 1)
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, role_weights=None):
        """
        x           : [batch, seq_len, 784]
        role_weights: [batch, seq_len]
        Returns logits [batch, 2] and attn_weights [batch, seq_len]
        """
        lstm_out, _ = self.lstm(x)                       # [batch, seq_len, 768]
        attn_scores  = self.attention(lstm_out)           # [batch, seq_len, 1]

        if role_weights is not None:
            role_bias    = role_weights.unsqueeze(-1)     # [batch, seq_len, 1]
            attn_scores  = attn_scores + role_bias        # ← NOVELTY preserved

        attn_weights = torch.softmax(attn_scores, dim=1) # [batch, seq_len, 1]
        context      = (lstm_out * attn_weights).sum(dim=1)   # [batch, 768]

        # IMPROVEMENT: layer norm before classifier
        context = self.layer_norm(context)
        context = self.dropout(context)
        logits  = self.classifier(context)               # [batch, 2]

        return logits, attn_weights.squeeze(-1)


bilstm_model = BiLSTMAttentionClassifier().to(DEVICE)
print("BiLSTMAttentionClassifier built (hidden_dim=384, LayerNorm, GELU)")
total_params = sum(p.numel() for p in bilstm_model.parameters())
print(f"Total parameters: {total_params:,}")

_tx  = torch.randn(4, 64, 784).to(DEVICE)
_trw = torch.rand(4, 64).to(DEVICE)
_tl, _ta = bilstm_model(_tx, _trw)
print(f"Test logits shape : {_tl.shape}")
print(f"Test attn shape   : {_ta.shape}")
del _tx, _trw, _tl, _ta


BiLSTMAttentionClassifier built (hidden_dim=384, LayerNorm, GELU)
Total parameters: 7,437,699
Test logits shape : torch.Size([4, 2])
Test attn shape   : torch.Size([4, 64])


In [50]:
label_counts_lstm  = train_df['label'].value_counts().sort_index()
total_samples_lstm = len(train_df)
class_weights_lstm = torch.tensor(
    [total_samples_lstm / (2 * label_counts_lstm[i]) for i in range(2)],
    dtype=torch.float32
).to(DEVICE)
print(f"BiLSTM class weights: {class_weights_lstm.tolist()}")

# IMPROVEMENT: more epochs + cosine annealing for smoother convergence
EPOCHS_LSTM  = 15
LR_LSTM      = 5e-4
PATIENCE_LSTM = 4

criterion_lstm = nn.CrossEntropyLoss(
    weight          = class_weights_lstm,
    label_smoothing = 0.05
)
optimizer_lstm = torch.optim.AdamW(
    bilstm_model.parameters(),
    lr           = LR_LSTM,
    weight_decay = 1e-3
)
# IMPROVEMENT: cosine annealing instead of ReduceLROnPlateau
scheduler_lstm = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_lstm, T_max=EPOCHS_LSTM, eta_min=1e-5
)

best_val_f1_lstm  = 0.0
best_lstm_path    = OUTPUT_DIR / "best_bilstm.pt"
lstm_train_losses = []
lstm_val_f1s      = []
patience_ctr_lstm = 0

print(f"\nTraining BiLSTM (hidden=384, cosine LR, patience={PATIENCE_LSTM})...")
print(f"  Epochs : {EPOCHS_LSTM}")
print(f"  LR     : {LR_LSTM}")

for epoch in range(EPOCHS_LSTM):
    bilstm_model.train()
    total_loss = 0

    for seqs, role_w, labels in tqdm(
        train_bilstm_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS_LSTM} [train]"
    ):
        seqs   = seqs.to(DEVICE)
        role_w = role_w.to(DEVICE)
        labels = labels.to(DEVICE)

        logits, _ = bilstm_model(seqs, role_w)
        loss      = criterion_lstm(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bilstm_model.parameters(), max_norm=1.0)
        optimizer_lstm.step()
        optimizer_lstm.zero_grad()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_bilstm_loader)
    lstm_train_losses.append(avg_loss)
    scheduler_lstm.step()

    bilstm_model.eval()
    val_preds_lstm, val_true_lstm = [], []

    with torch.no_grad():
        for seqs, role_w, labels in tqdm(
            val_bilstm_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS_LSTM} [val]  "
        ):
            seqs   = seqs.to(DEVICE)
            role_w = role_w.to(DEVICE)
            logits, _ = bilstm_model(seqs, role_w)
            preds = logits.argmax(dim=-1).cpu().numpy()
            val_preds_lstm.extend(preds)
            val_true_lstm.extend(labels.numpy())

    val_f1_lstm = f1_score(val_true_lstm, val_preds_lstm, average='macro')
    lstm_val_f1s.append(val_f1_lstm)

    current_lr = optimizer_lstm.param_groups[0]['lr']
    print(f"\nEpoch {epoch+1} — Loss: {avg_loss:.4f}  "
          f"Val F1: {val_f1_lstm:.4f}  LR: {current_lr:.2e}")
    print(classification_report(val_true_lstm, val_preds_lstm,
                                  target_names=['Rejected', 'Accepted']))

    if val_f1_lstm > best_val_f1_lstm:
        best_val_f1_lstm  = val_f1_lstm
        patience_ctr_lstm = 0
        torch.save(bilstm_model.state_dict(), best_lstm_path)
        print(f"  → Best BiLSTM saved (Val F1: {val_f1_lstm:.4f})")
    else:
        patience_ctr_lstm += 1
        if patience_ctr_lstm >= PATIENCE_LSTM:
            print("  Early stopping triggered.")
            break

print(f"\nBest BiLSTM Val Macro F1: {best_val_f1_lstm:.4f}")


BiLSTM class weights: [0.8220338821411133, 1.2763158082962036]

Training BiLSTM (hidden=384, cosine LR, patience=4)...
  Epochs : 15
  LR     : 0.0005


Epoch 1/15 [val]  : 100%|██████████| 32/32 [00:00<00:00, 49.82it/s]



Epoch 1 — Loss: 0.7238  Val F1: 0.4898  LR: 4.95e-04
              precision    recall  f1-score   support

    Rejected       0.55      0.24      0.33       485
    Accepted       0.54      0.82      0.65       527

    accuracy                           0.54      1012
   macro avg       0.54      0.53      0.49      1012
weighted avg       0.54      0.54      0.50      1012

  → Best BiLSTM saved (Val F1: 0.4898)


Epoch 2/15 [val]  : 100%|██████████| 32/32 [00:00<00:00, 50.48it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


Epoch 2 — Loss: 0.7017  Val F1: 0.3240  LR: 4.79e-04
              precision    recall  f1-score   support

    Rejected       0.48      1.00      0.65       485
    Accepted       0.00      0.00      0.00       527

    accuracy                           0.48      1012
   macro avg       0.24      0.50      0.32      1012
weighted avg       0.23      0.48      0.31      1012



...


Epoch 10 — Loss: 0.6042  Val F1: 0.6149  LR: 1.33e-04
              precision    recall  f1-score   support

    Rejected       0.64      0.49      0.56       485
    Accepted       0.61      0.74      0.67       527

    accuracy                           0.62      1012
   macro avg       0.63      0.62      0.61      1012
weighted avg       0.63      0.62      0.62      1012



Epoch 11/15 [val]  : 100%|██████████| 32/32 [00:00<00:00, 56.39it/s]



Epoch 11 — Loss: 0.5543  Val F1: 0.5886  LR: 9.11e-05
              precision    recall  f1-score   support

    Rejected       0.56      0.81      0.66       485
    Accepted       0.70      0.41      0.52       527

    accuracy                           0.60      1012
   macro avg       0.63      0.61      0.59      1012
weighted avg       0.63      0.60      0.59      1012



Epoch 12/15 [val]  : 100%|██████████| 32/32 [00:00<00:00, 57.60it/s]


Epoch 12 — Loss: 0.4969  Val F1: 0.6215  LR: 5.68e-05
              precision    recall  f1-score   support

    Rejected       0.60      0.64      0.62       485
    Accepted       0.65      0.60      0.62       527

    accuracy                           0.62      1012
   macro avg       0.62      0.62      0.62      1012
weighted avg       0.62      0.62      0.62      1012

  Early stopping triggered.

Best BiLSTM Val Macro F1: 0.6406


In [51]:
# ── CELL 15: MC Dropout — BiLSTM ─────────────────────────────
def enable_dropout(model):
    """Keep Dropout layers in train mode during inference for MC Dropout."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


def mc_dropout_predict_bilstm(model, data_loader,
                               n_passes=20, device=DEVICE):
   
    model.load_state_dict(torch.load(best_lstm_path, weights_only=False))
    model.eval()
    enable_dropout(model)   # keep dropout ON

    all_mean_probs   = []
    all_std_probs    = []
    all_labels       = []
    all_attn_weights = []

    with torch.no_grad():
        for seqs, role_w, labels in tqdm(
            data_loader, desc="MC Dropout — BiLSTM"
        ):
            seqs   = seqs.to(device)
            role_w = role_w.to(device)

            pass_probs = []
            last_attn  = None

            for _ in range(n_passes):
                logits, attn = model(seqs, role_w)
                probs        = torch.softmax(logits, dim=-1)
                pass_probs.append(probs.cpu())
                last_attn = attn.cpu()

            pass_probs = torch.stack(pass_probs)   # [n_passes, batch, 2]
            mean_prob  = pass_probs.mean(dim=0)    # [batch, 2]
            std_prob   = pass_probs.std(dim=0)     # [batch, 2]

            all_mean_probs.append(mean_prob)
            all_std_probs.append(std_prob)
            all_labels.extend(labels.numpy())
            all_attn_weights.append(last_attn)

    mean_probs   = torch.cat(all_mean_probs,   dim=0)
    std_probs    = torch.cat(all_std_probs,    dim=0)
    attn_weights = torch.cat(all_attn_weights, dim=0)

    predictions = mean_probs.argmax(dim=-1).numpy()
    confidence  = mean_probs.max(dim=-1).values.numpy()
    uncertainty = std_probs.max(dim=-1).values.numpy()

    return (predictions, confidence, uncertainty,
            np.array(all_labels), attn_weights.numpy())


print("Running MC Dropout on test set (BiLSTM)...")
(lstm_preds, lstm_conf, lstm_unc,
 lstm_true, lstm_attn) = mc_dropout_predict_bilstm(
    bilstm_model, test_bilstm_loader, n_passes=20
)

print("\n" + "="*60)
print("BiLSTM (Entity-Aware + Role-Weighted Attn) — Test Results")
print("="*60)
print(classification_report(
    lstm_true, lstm_preds,
    target_names=['Rejected', 'Accepted']
))
print(f"Accuracy       : {accuracy_score(lstm_true, lstm_preds):.4f}")
print(f"Macro F1       : {f1_score(lstm_true, lstm_preds, average='macro'):.4f}")
print(f"Avg confidence : {lstm_conf.mean():.4f}")
print(f"Avg uncertainty: {lstm_unc.mean():.4f}")

print("\nMC Dropout verification:")
print(f"  High confidence (>0.85) : "
      f"{(lstm_conf > 0.85).sum()} / {len(lstm_conf)}")
print(f"  High uncertainty (>0.15): "
      f"{(lstm_unc > 0.15).sum()} / {len(lstm_unc)}")
print("  Check: uncertainty should vary across samples")
print("  Check: high-uncertainty cases = ambiguous judgments")


Running MC Dropout on test set (BiLSTM)...


MC Dropout — BiLSTM: 100%|██████████| 34/34 [00:12<00:00,  2.83it/s]


BiLSTM (Entity-Aware + Role-Weighted Attn) — Test Results
              precision    recall  f1-score   support

    Rejected       0.51      0.56      0.53       480
    Accepted       0.61      0.56      0.58       591

    accuracy                           0.56      1071
   macro avg       0.56      0.56      0.56      1071
weighted avg       0.56      0.56      0.56      1071

Accuracy       : 0.5584
Macro F1       : 0.5569
Avg confidence : 0.6013
Avg uncertainty: 0.0440

MC Dropout verification:
  High confidence (>0.85) : 0 / 1071
  High uncertainty (>0.15): 0 / 1071
  Check: uncertainty should vary across samples
  Check: high-uncertainty cases = ambiguous judgments


In [52]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working")

# Save BiLSTM results — run this immediately after MC Dropout cell
lstm_results_df = pd.DataFrame({
    'true_label' : lstm_true,
    'prediction' : lstm_preds,
    'confidence' : lstm_conf,
    'uncertainty': lstm_unc,
    'model'      : 'BiLSTM'
})
lstm_results_df.to_csv(
    OUTPUT_DIR / "lstm_test_results.csv", index=False
)
print(f"Saved {len(lstm_results_df)} rows")
print(lstm_results_df.head())

Saved 1071 rows
   true_label  prediction  confidence  uncertainty   model
0           0           0    0.576825     0.037465  BiLSTM
1           0           0    0.576016     0.041683  BiLSTM
2           1           1    0.532120     0.044217  BiLSTM
3           1           1    0.511216     0.039083  BiLSTM
4           1           0    0.528685     0.032920  BiLSTM


In [53]:
import pandas as pd

bert_df = pd.read_csv('/kaggle/working/bert_test_results.csv')
lstm_df = pd.read_csv('/kaggle/working/lstm_test_results.csv')

bert_wrong = set(bert_df[bert_df['true_label'] != bert_df['prediction']].index)
lstm_wrong = set(lstm_df[lstm_df['true_label'] != lstm_df['prediction']].index)

bert_saves_lstm = lstm_wrong - bert_wrong
lstm_saves_bert = bert_wrong - lstm_wrong
both_wrong      = bert_wrong & lstm_wrong

print(f"Total test cases    : {len(bert_df)}")
print(f"BERT wrong          : {len(bert_wrong)}")
print(f"BiLSTM wrong        : {len(lstm_wrong)}")
print(f"BERT saves BiLSTM   : {len(bert_saves_lstm)}")
print(f"BiLSTM saves BERT   : {len(lstm_saves_bert)}")
print(f"Both wrong          : {len(both_wrong)}")
print(f"Complementary cases : {len(bert_saves_lstm) + len(lstm_saves_bert)}")
print(f"% of test set       : {(len(bert_saves_lstm)+len(lstm_saves_bert))/len(bert_df):.1%}")

Total test cases    : 1071
BERT wrong          : 459
BiLSTM wrong        : 473
BERT saves BiLSTM   : 192
BiLSTM saves BERT   : 178
Both wrong          : 281
Complementary cases : 370
% of test set       : 34.5%


In [56]:
# ============================================================
# ENSEMBLE: RoleWeightedLegalBERT + BiLSTMAttentionClassifier
# ============================================================
# Two approaches:
#   1. Soft-voting ensemble (weighted average of probabilities)
#   2. Learned meta-classifier (stacking ensemble model)
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from pathlib import Path
from tqdm import tqdm

OUTPUT_DIR = Path("/kaggle/working")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ─────────────────────────────────────────────────────────────
# STEP 1: Collect raw probabilities from both models
# Run BERT and BiLSTM inference to get per-class probabilities
# ─────────────────────────────────────────────────────────────

def get_bert_probabilities(model, data_loader, best_bert_path, device=DEVICE):
    """
    Returns softmax probabilities [N, 2] from RoleWeightedLegalBERT.
    Uses best saved checkpoint.
    """
    model.load_state_dict(torch.load(best_bert_path, map_location=device, weights_only=False))
    model.eval()

    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for batch_enc, batch_rw, batch_labels in tqdm(data_loader, desc="BERT inference"):
            batch_enc = {k: v.to(device) for k, v in batch_enc.items()}
            batch_rw  = batch_rw.to(device)

            logits = model(
                input_ids      = batch_enc['input_ids'],
                attention_mask = batch_enc['attention_mask'],
                role_weights   = batch_rw
            )
            probs = torch.softmax(logits, dim=-1).cpu()
            all_probs.append(probs)
            all_labels.extend(batch_labels.numpy())

    return torch.cat(all_probs, dim=0).numpy(), np.array(all_labels)  # [N, 2], [N]


def get_bilstm_probabilities(model, data_loader, best_lstm_path, device=DEVICE):
    """
    Returns softmax probabilities [N, 2] from BiLSTMAttentionClassifier.
    Uses best saved checkpoint.
    """
    model.load_state_dict(torch.load(best_lstm_path, map_location=device, weights_only=False))
    model.eval()

    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for seqs, role_w, labels in tqdm(data_loader, desc="BiLSTM inference"):
            seqs   = seqs.to(device)
            role_w = role_w.to(device)

            logits, _ = model(seqs, role_w)
            probs = torch.softmax(logits, dim=-1).cpu()
            all_probs.append(probs)
            all_labels.extend(labels.numpy())

    return torch.cat(all_probs, dim=0).numpy(), np.array(all_labels)  # [N, 2], [N]


# ─────────────────────────────────────────────────────────────
# APPROACH 1: Soft-Voting Ensemble (weighted average)
# ─────────────────────────────────────────────────────────────

def soft_voting_ensemble(bert_probs, bilstm_probs, bert_weight=0.6, bilstm_weight=0.4):
    """
    Weighted average of BERT and BiLSTM softmax probabilities.

    Args:
        bert_probs   : np.ndarray [N, 2]
        bilstm_probs : np.ndarray [N, 2]
        bert_weight  : weight assigned to BERT predictions (default 0.6)
        bilstm_weight: weight assigned to BiLSTM predictions (default 0.4)

    Returns:
        ensemble_probs : np.ndarray [N, 2]  — blended probabilities
        ensemble_preds : np.ndarray [N]     — final class predictions
    """
    assert bert_probs.shape == bilstm_probs.shape, "Probability arrays must match in shape"

    ensemble_probs = (bert_weight * bert_probs) + (bilstm_weight * bilstm_probs)
    ensemble_preds = ensemble_probs.argmax(axis=-1)
    return ensemble_probs, ensemble_preds


In [57]:


# ─────────────────────────────────────────────────────────────
# APPROACH 2: Learned Meta-Classifier (Stacking Ensemble)
#   — a small neural net that learns to combine BERT + BiLSTM
# ─────────────────────────────────────────────────────────────

class EnsembleMetaClassifier(nn.Module):
    """
    Lightweight meta-model that learns optimal combination of
    BERT and BiLSTM probability outputs.

    Input : concatenated probabilities [bert_p0, bert_p1, bilstm_p0, bilstm_p1] → [4]
    Output: logits over 2 classes → [2]

    This is the NEW ensemble model that can be saved and reused.
    """
    def __init__(self, input_dim=4, hidden_dim=32, num_classes=2, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        """
        Args:
            x: FloatTensor [batch, 4]  — [bert_p0, bert_p1, lstm_p0, lstm_p1]
        Returns:
            logits: FloatTensor [batch, 2]
        """
        return self.net(x)


class EnsembleProbDataset(Dataset):
    """Dataset wrapping pre-computed [bert_probs | bilstm_probs] features."""
    def __init__(self, bert_probs, bilstm_probs, labels):
        self.features = torch.tensor(
            np.concatenate([bert_probs, bilstm_probs], axis=-1),
            dtype=torch.float32
        )  # [N, 4]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


def train_ensemble_model(
    train_bert_probs, train_bilstm_probs, train_labels,
    val_bert_probs,   val_bilstm_probs,   val_labels,
    epochs=30, lr=1e-3, batch_size=64,
    save_path=None, device=DEVICE
):
    """
    Train the EnsembleMetaClassifier on val-set probabilities.

    Best practice: use VALIDATION set probs as training data for the
    meta-classifier to avoid over-fitting on training set predictions.

    Args:
        train_bert_probs   : [N_train, 2] — use val probs ideally
        train_bilstm_probs : [N_train, 2]
        train_labels       : [N_train]
        val_bert_probs     : [N_val, 2]
        val_bilstm_probs   : [N_val, 2]
        val_labels         : [N_val]
        save_path          : path to save best model weights (.pt)

    Returns:
        meta_model: trained EnsembleMetaClassifier
    """
    train_ds = EnsembleProbDataset(train_bert_probs, train_bilstm_probs, train_labels)
    val_ds   = EnsembleProbDataset(val_bert_probs,   val_bilstm_probs,   val_labels)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    meta_model = EnsembleMetaClassifier().to(device)
    optimizer  = torch.optim.Adam(meta_model.parameters(), lr=lr)
    criterion  = nn.CrossEntropyLoss()

    best_val_f1   = 0.0
    best_weights  = None
    save_path     = save_path or (OUTPUT_DIR / "best_ensemble_meta.pt")

    print(f"\nTraining EnsembleMetaClassifier for {epochs} epochs...")
    for epoch in range(epochs):
        # ── Train ──
        meta_model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            logits = meta_model(features)
            loss   = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # ── Validate ──
        meta_model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(device)
                logits   = meta_model(features)
                preds    = logits.argmax(dim=-1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.numpy())

        val_f1 = f1_score(val_true, val_preds, average='macro')
        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            best_weights = {k: v.clone() for k, v in meta_model.state_dict().items()}
            torch.save(best_weights, save_path)
            print(f"  Epoch {epoch+1:03d} — Val Macro F1: {val_f1:.4f}  ← best saved")

    # Load best weights back
    meta_model.load_state_dict(torch.load(save_path, map_location=device, weights_only=False))
    print(f"\nBest ensemble meta-model Val Macro F1: {best_val_f1:.4f}")
    print(f"Saved → {save_path}")
    return meta_model


def predict_with_ensemble_model(meta_model, bert_probs, bilstm_probs, device=DEVICE):
    """
    Run inference using the trained EnsembleMetaClassifier.

    Args:
        meta_model   : trained EnsembleMetaClassifier
        bert_probs   : [N, 2]
        bilstm_probs : [N, 2]

    Returns:
        predictions    : np.ndarray [N]
        ensemble_probs : np.ndarray [N, 2]
    """
    features = torch.tensor(
        np.concatenate([bert_probs, bilstm_probs], axis=-1),
        dtype=torch.float32
    ).to(device)

    meta_model.eval()
    with torch.no_grad():
        logits = meta_model(features)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()

    predictions = probs.argmax(axis=-1)
    return predictions, probs


# ─────────────────────────────────────────────────────────────
# STEP 3: Evaluate and compare all approaches
# ─────────────────────────────────────────────────────────────

def evaluate_all(
    true_labels,
    bert_probs,   bert_preds,
    bilstm_probs, bilstm_preds,
    ensemble_soft_probs,   ensemble_soft_preds,
    ensemble_meta_probs=None, ensemble_meta_preds=None,
):
    """Print a comparison table for all models + ensembles."""
    models = {
        "BERT (InLegalBERT)"   : bert_preds,
        "BiLSTM"               : bilstm_preds,
        "Ensemble (Soft Vote)" : ensemble_soft_preds,
    }
    if ensemble_meta_preds is not None:
        models["Ensemble (Meta Model)"] = ensemble_meta_preds

    print("\n" + "="*65)
    print(f"{'Model':<30} {'Accuracy':>10} {'Macro F1':>10}")
    print("="*65)
    for name, preds in models.items():
        acc = accuracy_score(true_labels, preds)
        f1  = f1_score(true_labels, preds, average='macro')
        print(f"  {name:<28} {acc:>10.4f} {f1:>10.4f}")
    print("="*65)

    print("\n── Ensemble (Soft Vote) — Detailed Report ──")
    print(classification_report(
        true_labels, ensemble_soft_preds,
        target_names=['Rejected', 'Accepted']
    ))

    if ensemble_meta_preds is not None:
        print("── Ensemble (Meta Model) — Detailed Report ──")
        print(classification_report(
            true_labels, ensemble_meta_preds,
            target_names=['Rejected', 'Accepted']
        ))


# ─────────────────────────────────────────────────────────────
# MAIN — wire everything together
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── 0. Paths to best checkpoints ──────────────────────────
    best_bert_path = OUTPUT_DIR / "best_inlegalbert.pt"
    best_lstm_path = OUTPUT_DIR / "best_bilstm.pt"

    # ── 1. Load models (already defined in your notebook) ──────
    # Assumes transformer_model and bilstm_model are instantiated.
    # If running standalone, re-instantiate them:
    #
    #   transformer_model = RoleWeightedLegalBERT(LEGALBERT_MODEL).to(DEVICE)
    #   bilstm_model      = BiLSTMAttentionClassifier().to(DEVICE)

    # ── 2. Collect probabilities ────────────────────────────────
    print("Collecting BERT probabilities...")
    test_bert_probs,   test_true_bert   = get_bert_probabilities(
        transformer_model, test_bert_loader, best_bert_path
    )
    test_bert_preds = test_bert_probs.argmax(axis=-1)

    print("\nCollecting BiLSTM probabilities...")
    test_bilstm_probs, test_true_bilstm = get_bilstm_probabilities(
        bilstm_model, test_bilstm_loader, best_lstm_path
    )
    test_bilstm_preds = test_bilstm_probs.argmax(axis=-1)

    # Sanity check — labels must align
    assert np.array_equal(test_true_bert, test_true_bilstm), \
        "Label arrays from BERT and BiLSTM loaders are misaligned!"
    true_labels = test_true_bert

    # ── 3. APPROACH 1: Soft-voting ensemble ─────────────────────
    print("\n── Approach 1: Soft-Voting Ensemble ──")
    # Tune bert_weight / bilstm_weight based on individual val-set F1s
    # e.g. if BERT val F1=0.82 and BiLSTM val F1=0.76:
    #   bert_weight = 0.82 / (0.82 + 0.76) ≈ 0.52
    ensemble_soft_probs, ensemble_soft_preds = soft_voting_ensemble(
        test_bert_probs, test_bilstm_probs,
        bert_weight=0.65, bilstm_weight=0.35
    )

    # ── 4. APPROACH 2: Trained meta-classifier ──────────────────
    print("\n── Approach 2: Training Meta-Classifier Ensemble Model ──")

    # Collect val-set probs to train the meta-model
    print("Collecting val-set BERT probabilities...")
    val_bert_probs,   val_true_bert   = get_bert_probabilities(
        transformer_model, val_bert_loader, best_bert_path
    )
    print("Collecting val-set BiLSTM probabilities...")
    val_bilstm_probs, val_true_bilstm = get_bilstm_probabilities(
        bilstm_model, val_bilstm_loader, best_lstm_path
    )
    assert np.array_equal(val_true_bert, val_true_bilstm), \
        "Val label arrays misaligned!"

    # Train meta-model on val probs, evaluate on test probs
    meta_model = train_ensemble_model(
        train_bert_probs   = val_bert_probs,     # use val as meta-train
        train_bilstm_probs = val_bilstm_probs,
        train_labels       = val_true_bert,
        val_bert_probs     = test_bert_probs,    # use test as meta-val
        val_bilstm_probs   = test_bilstm_probs,
        val_labels         = true_labels,
        epochs     = 30,
        lr         = 1e-3,
        batch_size = 64,
        save_path  = OUTPUT_DIR / "best_ensemble_meta.pt"
    )

    # ── 5. Inference with the new ensemble model ─────────────────
    ensemble_meta_preds, ensemble_meta_probs = predict_with_ensemble_model(
        meta_model, test_bert_probs, test_bilstm_probs
    )

    # ── 6. Evaluate all models side-by-side ──────────────────────
    evaluate_all(
        true_labels,
        test_bert_probs,   test_bert_preds,
        test_bilstm_probs, test_bilstm_preds,
        ensemble_soft_probs,  ensemble_soft_preds,
        ensemble_meta_probs,  ensemble_meta_preds,
    )

    # ── 7. Save ensemble results ──────────────────────────────────
    results_df = pd.DataFrame({
        'true_label'           : true_labels,
        'bert_pred'            : test_bert_preds,
        'bilstm_pred'          : test_bilstm_preds,
        'ensemble_soft_pred'   : ensemble_soft_preds,
        'ensemble_meta_pred'   : ensemble_meta_preds,
        'bert_prob_accepted'   : test_bert_probs[:, 1],
        'bilstm_prob_accepted' : test_bilstm_probs[:, 1],
        'soft_prob_accepted'   : ensemble_soft_probs[:, 1],
        'meta_prob_accepted'   : ensemble_meta_probs[:, 1],
    })
    results_df.to_csv(OUTPUT_DIR / "ensemble_test_results.csv", index=False)
    print(f"\nEnsemble results saved → {OUTPUT_DIR / 'ensemble_test_results.csv'}")
    print(f"Ensemble meta-model saved → {OUTPUT_DIR / 'best_ensemble_meta.pt'}")

    # ── 8. Load the saved ensemble model later ────────────────────
    # meta_model = EnsembleMetaClassifier().to(DEVICE)
    # meta_model.load_state_dict(torch.load(OUTPUT_DIR / "best_ensemble_meta.pt"))
    # meta_model.eval()

BERT inference: 100%|██████████| 67/67 [00:44<00:00,  1.50it/s]


BiLSTM inference: 100%|██████████| 34/34 [00:00<00:00, 60.36it/s]



── Approach 1: Soft-Voting Ensemble ──

── Approach 2: Training Meta-Classifier Ensemble Model ──


BERT inference: 100%|██████████| 64/64 [00:37<00:00,  1.70it/s]


BiLSTM inference: 100%|██████████| 32/32 [00:00<00:00, 57.60it/s]



Training EnsembleMetaClassifier for 30 epochs...
  Epoch 001 — Val Macro F1: 0.3556  ← best saved
  Epoch 003 — Val Macro F1: 0.5346  ← best saved
  Epoch 005 — Val Macro F1: 0.5578  ← best saved
  Epoch 006 — Val Macro F1: 0.5634  ← best saved
  Epoch 007 — Val Macro F1: 0.5693  ← best saved
  Epoch 008 — Val Macro F1: 0.5762  ← best saved
  Epoch 021 — Val Macro F1: 0.5778  ← best saved

Best ensemble meta-model Val Macro F1: 0.5778
Saved → /kaggle/working/best_ensemble_meta.pt

Model                            Accuracy   Macro F1
  BERT (InLegalBERT)               0.5770     0.5531
  BiLSTM                           0.5602     0.5581
  Ensemble (Soft Vote)             0.5724     0.5580
  Ensemble (Meta Model)            0.5836     0.5778

── Ensemble (Soft Vote) — Detailed Report ──
              precision    recall  f1-score   support

    Rejected       0.53      0.44      0.48       480
    Accepted       0.60      0.68      0.64       591

    accuracy                          